In [ ]:
# Llama-3.2 is gated. Store a Kaggle secret named HF_TOKEN.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Hugging Face login: OK')
except Exception as exc:
    print('HF login was not completed automatically:', exc)
    print('If the model is already cached and accessible, you can continue.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:
"""

Hint:
# Analyze the weights of the query projection layer (q_proj) where layer = k
 q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

# Apply SVD to decompose the weight matrix into U (left singular vectors), S (singular values), and V (right singular vectors).
 U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

# Compute the retained energy ratio (Cumulative energy) of each singular value.
energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
"""

import pandas as pd

# Layers required by Question 4
layers_to_analyze = [0, 7, 15]
results_ranks = {}
svd_data = {}

print("Performing SVD analysis on layers 0, 7, and 15...")

for k in layers_to_analyze:
  # 1. Extract weights of the query projection layer (q_proj)
  q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

  # 2. Apply Full SVD (compute on float32 for numerical stability)
  U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

  # 3. Compute cumulative energy ratio (%) using squared singular values
  energy_ratios = torch.cumsum(S**2, dim=0) / torch.sum(S**2) * 100

  # 4. Find the minimum target rank required to retain at least 50% cumulative energy
  # (indices are 0-based, so rank = index + 1)
  min_rank_50 = (energy_ratios >= 50.0).nonzero(as_tuple=True)[0][0].item() + 1
  results_ranks[f"Layer {k}"] = min_rank_50

  svd_data[k] = {
      "S": S.cpu().numpy(),
      "energy_ratios": energy_ratios.cpu().numpy(),
  }

# Display the summary table required for Question 4b
df_ranks = pd.DataFrame(
    [results_ranks],
    index=["Minimum Target Rank for query projection weights"],
)
print("\n" + "=" * 65)
print("QUESTION 4b TABLE: Minimum Target Rank for >= 50% Cumulative Energy")
print("=" * 65)
display(df_ranks)

In [ ]:
# Generate visualizations for Layer 0, 7, 15 (First 200 singular values)
# Total: 6 charts (1 bar chart and 1 line chart per layer)
for k in layers_to_analyze:
  S_200 = svd_data[k]["S"][:200]
  energy_200 = svd_data[k]["energy_ratios"][:200]
  indices = list(range(1, 201))

  # --- Chart 1: Bar chart for Singular Value Magnitude ---
  plt.figure(figsize=(7, 4.5))
  plt.bar(indices, S_200, width=1.0, color="#1f77b4", edgecolor="none")
  plt.title(f"Layer {k} - Singular Value Magnitude", fontsize=12)
  plt.xlabel("Singular Value Index", fontsize=10)
  plt.ylabel("Singular Value Magnitude", fontsize=10)
  plt.xlim(0, 200)
  plt.grid(True, linestyle="--", alpha=0.5)
  plt.tight_layout()
  plt.savefig(f"layer_{k}_singular_value_magnitude.png", dpi=300)
  plt.show()

  # --- Chart 2: Line chart for Cumulative Energy (%) ---
  plt.figure(figsize=(7, 4.5))
  plt.plot(indices, energy_200, color="#1f77b4", linewidth=2)
  plt.axhline(
      y=50,
      color="red",
      linestyle="--",
      label="50% Threshold",
  )
  plt.axvline(
      x=results_ranks[f"Layer {k}"],
      color="green",
      linestyle=":",
      label=f"Min Rank = {results_ranks[f'Layer {k}']}",
  )
  plt.title(f"Layer {k} - Cumulative Energy (%)", fontsize=12)
  plt.xlabel("Target Rank", fontsize=10)
  plt.ylabel("Cumulative Energy (%)", fontsize=10)
  plt.xlim(0, 200)
  plt.ylim(0, 100)
  plt.legend(loc="lower right")
  plt.grid(True, linestyle="--", alpha=0.5)
  plt.tight_layout()
  plt.savefig(f"layer_{k}_cumulative_energy.png", dpi=300)
  plt.show()

print("\nSuccessfully generated and saved all 6 charts as high-resolution PNGs.")